# RT Notebook 14: Generic Topology Feature Ablation

Purpose: test whether Notebook 13 v2 classification survives removal of near-definitional branch-pair and merge-depth features.

Claim ceiling before governed output induction: `C1_SPECIFICATION_ONLY`.


## 1. Bootstrap

Upload or provide `NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip` at `/content/`, Drive, or the local repo path.


In [ ]:
from __future__ import annotations

import json, zipfile
from pathlib import Path
from typing import Dict, List

import pandas as pd

SEED = 140014
SPEC_ID = 'NB14_GENERIC_TOPOLOGY_FEATURE_ABLATION_001'
ZIP_CANDIDATES = [
    Path('/content/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
    Path('/content/drive/MyDrive/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
    Path('departments/colab/results/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
]
NB13_ZIP = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if NB13_ZIP is None:
    raise FileNotFoundError('Could not find NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip')

OUTPUT_DIR = Path('/content') / SPEC_ID if Path('/content').exists() else Path('departments/colab/results') / SPEC_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORBIDDEN_FEATURES = [
    'root_branch_pair_count',
    'irreversible_separated_pair_count',
    'has_irreversible_separation',
    'merge_depth_min',
    'merge_depth_mean',
    'merge_depth_max',
]

FEATURE_FAMILIES: Dict[str, List[str]] = {
    'graph_size': ['node_count', 'edge_count'],
    'reachability_lattice': ['reachability_closure_count', 'reachability_collision_count', 'closure_min_size', 'closure_max_size', 'closure_lattice_node_count'],
    'scc_structure': ['scc_count'],
    'condensation_dag': ['condensation_node_count', 'condensation_edge_count', 'condensation_source_count', 'condensation_sink_count'],
    'basin_decomposition': ['terminal_basin_count'],
    'articulation_hierarchy': ['articulation_point_count', 'articulation_min_depth', 'articulation_max_depth'],
    'bridge_structure': ['bridge_count'],
    'partial_order_width': ['partial_order_width', 'partial_order_width_exact'],
    'dominance_tree': ['dominance_tree_node_count', 'dominance_tree_max_depth'],
}

print('Using NB13 zip:', NB13_ZIP)
print('Output dir:', OUTPUT_DIR)


## 2. Load Notebook 13 v2 Invariants


In [ ]:
with zipfile.ZipFile(NB13_ZIP) as zf:
    with zf.open('topological_invariants.parquet') as f:
        invariants = pd.read_parquet(f)
    nb13_manifest = json.loads(zf.read('manifest.json').decode('utf-8'))

required = ['geometry', *FORBIDDEN_FEATURES]
missing = [c for c in required if c not in invariants.columns]
if missing:
    raise KeyError(f'NB13 invariant table missing required columns: {missing}')

print('Rows:', len(invariants))
print(invariants['geometry'].value_counts(dropna=False))
nb13_manifest


## 3. Build Permitted Feature Sets

Generic feature sets must exclude all forbidden near-definitional branch-pair and merge-depth features.


In [ ]:
numeric_columns = [c for c in invariants.columns if pd.api.types.is_numeric_dtype(invariants[c])]
label_columns = {'config_id', 'geometry', 'J'}
all_generic_features = [
    c for family in FEATURE_FAMILIES.values() for c in family
    if c in numeric_columns and c not in label_columns and c not in FORBIDDEN_FEATURES
]
forbidden_present = [c for c in FORBIDDEN_FEATURES if c in all_generic_features]
if forbidden_present:
    raise AssertionError(f'Forbidden features leaked into generic feature set: {forbidden_present}')

feature_manifest = {
    'spec_id': SPEC_ID,
    'forbidden_features': FORBIDDEN_FEATURES,
    'feature_families': FEATURE_FAMILIES,
    'all_generic_features': all_generic_features,
    'graph_size_only_features': [c for c in FEATURE_FAMILIES['graph_size'] if c in numeric_columns],
    'forbidden_proxy_only_features': [c for c in FORBIDDEN_FEATURES if c in numeric_columns],
}
(OUTPUT_DIR / 'feature_manifest.json').write_text(json.dumps(feature_manifest, indent=2), encoding='utf-8')
feature_manifest


## 4. Classifier Helper


In [ ]:
def evaluate_feature_set(name: str, features: List[str]) -> Dict[str, object]:
    if not features:
        return {'condition': name, 'status': 'SKIPPED_EMPTY_FEATURE_SET', 'feature_count': 0}
    try:
        from sklearn.dummy import DummyClassifier
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import balanced_accuracy_score, classification_report
        from sklearn.model_selection import train_test_split
        X = invariants[features].fillna(-1)
        y = invariants['geometry']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
        dummy = DummyClassifier(strategy='most_frequent')
        dummy.fit(X_train, y_train)
        model = RandomForestClassifier(n_estimators=160, random_state=SEED, class_weight='balanced', n_jobs=-1)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        return {
            'condition': name,
            'status': 'EXECUTED',
            'feature_count': len(features),
            'features': features,
            'dummy_balanced_accuracy': float(balanced_accuracy_score(y_test, dummy.predict(X_test))),
            'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
            'classification_report': classification_report(y_test, pred, output_dict=True),
        }
    except Exception as exc:
        return {'condition': name, 'status': 'FAILED', 'feature_count': len(features), 'features': features, 'error': repr(exc)}


## 5. Run Generic Baselines and Feature-Family Ablations


In [ ]:
reports = []
reports.append(evaluate_feature_set('all_generic', all_generic_features))
reports.append(evaluate_feature_set('graph_size_only', feature_manifest['graph_size_only_features']))
reports.append(evaluate_feature_set('forbidden_proxy_only_diagnostic', feature_manifest['forbidden_proxy_only_features']))

all_set = set(all_generic_features)
for family, cols in FEATURE_FAMILIES.items():
    retained = sorted(all_set - {c for c in cols if c in all_set})
    reports.append(evaluate_feature_set(f'remove_{family}', retained))

all_generic_score = next((r.get('balanced_accuracy') for r in reports if r['condition'] == 'all_generic' and r['status'] == 'EXECUTED'), None)
rows = []
for r in reports:
    score = r.get('balanced_accuracy')
    rows.append({
        'condition': r['condition'],
        'status': r['status'],
        'feature_count': r.get('feature_count'),
        'balanced_accuracy': score,
        'delta_from_all_generic': None if score is None or all_generic_score is None else float(all_generic_score - score),
        'dummy_balanced_accuracy': r.get('dummy_balanced_accuracy')
    })
ablation = pd.DataFrame(rows)
ablation.to_csv(OUTPUT_DIR / 'feature_ablation_results.csv', index=False)
(OUTPUT_DIR / 'generic_topology_classification_report.json').write_text(json.dumps({'spec_id': SPEC_ID, 'reports': reports}, indent=2), encoding='utf-8')
ablation


## 6. Manifest


In [ ]:
manifest = {
    'notebook': 'RT Notebook 14',
    'title': 'Generic Topology Feature Ablation',
    'spec_id': SPEC_ID,
    'seed': SEED,
    'source_archive': str(NB13_ZIP),
    'rows_loaded': int(len(invariants)),
    'forbidden_features': FORBIDDEN_FEATURES,
    'output_files': [
        'feature_manifest.json',
        'feature_ablation_results.csv',
        'generic_topology_classification_report.json'
    ],
    'claim_ceiling': 'C1 before governed output induction; C2 candidate only after result zip registration and review',
    'interpretation_constraint': 'Bounded to Notebook 13 v2 invariant table and Notebook 12 source domain; no external physical validation.'
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest
